# Ansys HFSS → AWS Palace: one-notebook pipeline

Run the cells **top to bottom**. At the end you have **one folder** containing
everything the HPC needs — drag it over and run `qsub run_palace.pbs`. Nothing
else to remember.

**Before you start:** HFSS must be open with your project loaded, and this
notebook must live in the pipeline folder (next to `export_for_palace.py`,
`mesh_any.py`, `palace_matchers.py`, `step_bodies.py`,
`write_palace_config.py`, `mesh_stats.py`).

| step | what happens | where |
|---|---|---|
| 1 | settings — the only cell you routinely edit | here |
| 2 | export geometry + physics from HFSS into a fresh run folder | your PC |
| 3 | review the mesh size for every body; adjust if needed | your PC |
| 4 | build the mesh (gmsh) + quality report | your PC |
| 5 | generate `palace_config.json` | your PC |
| 6 | generate `run_palace.pbs` + completeness check | your PC |
| 7 | copy folder to HPC, `qsub` | HPC |

In [22]:
# ============================== 1. SETTINGS ==============================
# The only cell you routinely edit.

DESIGN_NAME = "cavity_pin_loss_chip_JJ"   # HFSS design to export (None = active design)
TAG         = "run3"                      # label stamped into mesh/config filenames

# --- mesh sizes (mm) you want to FORCE, overriding whatever HFSS has.
# Leave empty {} to use the design's own mesh operations / stats.
# Consult mesh_size_database.json for the lab's tested values.
#   0.0002 = 0.2 um, 0.05 = 50 um, 1.0 = 1 mm
SIZES = {
    "JJ":          0.0002,   # 0.2 um — lab standard, geometrically tiny so it's cheap; the most sensitive object, don't coarsen
    "thin_lead":   0.0005,   # 0.5 um — 2 elements across its 1 um width; the 0.2 um default was ~half your tet budget
    "medium_lead": 0.003,    # 3 um — same as the HFSS op; stated for the record
    "pads":        0.20,     # the memory fix: 0.05 was ~10^5 tets in the 10s grading slab; AMR re-refines if the field cares
    "pin":         1.0,      # PEC pin; its curved wall is curvature-capped at ~0.39 mm anyway
    "Pin_1":       0.4,      # copper pin = a loss surface, worth keeping moderately fine
    "cavity":      2.0,      # bulk size and the global ceiling grading relaxes to; AMR owns bulk refinement
}

# --- mesher options
MESH_ORDER = 2        # 2 = curvilinear (curved pins meshed exactly), 1 = straight
CURVATURE  = 24       # elements per 2*pi of curvature (8 is fine at order 2)

# --- Palace solver options (written into the config)
PALACE = {
    "target_freq_GHz": 3.5,   # keep BELOW the lowest expected mode
    "n_modes": 5,
    "amr_max_its": 3,
    "amr_max_size": 2500000,  # memory guard: ~ (HPC mem_gb / 0.06) unknowns
}

# --- HPC job parameters (for run_palace.pbs)
HPC = {
    "jobname": f"palace_{TAG}",
    "ncpus": 36,
    "mem_gb": 240,
    "walltime": "06:00:00",
    "palace_bin": "$HOME/palace_build_openmpi/bin/palace-x86_64.bin",
}

import os, pipeline_helpers as ph
print("settings loaded")

settings loaded


## 2. Export from HFSS
Creates a fresh timestamped run folder next to your `.aedt` file with
`device.step`, `device_config.json`, and copies of the pipeline scripts.
HFSS stays open; nothing in your project is modified.

In [23]:
import sys
!{sys.executable} -m pip install pyaedt

In [24]:
import export_for_palace as exp

RUN_DIR = exp.main(
    design_name_arg=DESIGN_NAME,
    mesh_size_overrides={},        # sizes are applied at MESH time (step 4),
                                   # so the exported config stays pristine
    palace_solver_overrides=PALACE,
    mesher_overrides={"mesh_order": MESH_ORDER,
                      "curvature_elements_per_2pi": CURVATURE},
)
print("\nRUN_DIR =", RUN_DIR)

export_for_palace.py
Connecting to running Ansys HFSS instance...
PyAEDT INFO: Python version 3.8.20 (default, Oct  3 2024, 15:19:54) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.5.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\qcrew5\AppData\Local\Temp\pyaedt_qcrew5_7e554975-651b-4715-9969-0090d15bce93.log is enabled.
PyAEDT INFO: Log on AEDT is enabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: Found active AEDT gRPC session on port 50052.
PyAEDT INFO: AEDT installation Path C:\Program Files\AnsysEM\v242\Win64
PyAEDT INFO: No project is defined. Project test_export exists and has been read.
PyAEDT INFO: Active Design set to cavity_pin_loss_chip_JJ
PyAEDT INFO: Aedt Objects correctly read

Project: test_export
Design:  cavity_pin_loss_chip_JJ
Path:    //Qcrew_drive/home/vashti_enjoy/
Output:  //Qcrew_dr

## 3. Review mesh sizes
One line per body: the size the mesh will use and where it comes from
(`operation` = set in HFSS, `stats` = HFSS's adapted mesh, `auto` = analytic
guess). Lab-database hints are shown where a body name matches a known
component. **If anything looks wrong, put the correction in `SIZES` in
cell 1 and re-run from this cell** — no need to re-export.

In [25]:
base_sizes = ph.size_report(
    RUN_DIR,
    database_path=os.path.join(os.path.dirname(os.path.abspath("__file__")),
                               "mesh_size_database.json"))
if SIZES:
    print("\nyour overrides for this run:")
    for b, s in SIZES.items():
        print(f"  {b:<16}{s:g} mm")

body            role                base size   source
------------------------------------------------------------------------------
JJ              junction            0.0002 mm   operation 'JJ_mesh_0.2um'
                                                [JJ: standard 0.0002, lab range 0.0001-0.0005] mm
Pin_1           conductor_solid     1.16128 mm  HFSS mesh stats (Setup4 RMS edge)
cavity          dielectric          0.66493 mm  HFSS mesh stats (Setup4 RMS edge)
chip            dielectric          0.150182 mm HFSS mesh stats (Setup4 RMS edge)
medium_lead     pec_sheet           0.003 mm    operation 'medium_3um'
                                                [coarse_lead: standard 0.003, lab range 0.001-0.05] mm
pads            pec_sheet           0.05 mm     operation 'pads_50um'
                                                [qubit_pads: lab range 0.01-0.1] mm
pin             pec_solid           0.5 mm      operation 'mesh_pin'
thin_lead       pec_sheet           0.0002 mm   ope

## 4. Mesh
Builds `device_<TAG>.msh` with your overrides, then prints the quality
report. Sanity targets: **tets ≲ 200k** for 3 AMR passes in ~240 GB;
worst aspect ratio should stay in single digits.

In [26]:
import sys
!{sys.executable} -m pip install gmsh

In [27]:
import subprocess, sys

cmd = [sys.executable, "mesh_any.py", "--config", "device_config.json",
       "--tag", TAG, "--mesh-order", str(MESH_ORDER),
       "--curvature-segments", str(CURVATURE)]
for body, mm in SIZES.items():
    cmd += ["--size", f"{body}={mm}"]

print("$", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, cwd=RUN_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"mesher failed (exit {proc.returncode})")

MSH, GROUPS = f"device_{TAG}.msh", f"device_groups_{TAG}.json"
ph.run_mesh_stats(RUN_DIR, MSH)

$ c:\Users\qcrew5\anaconda3\envs\qiskit_metal\python.exe mesh_any.py --config device_config.json --tag run3 --mesh-order 2 --curvature-segments 24 --size JJ=0.0002 --size thin_lead=0.0005 --size medium_lead=0.003 --size pads=0.2 --size pin=1.0 --size Pin_1=0.4 --size cavity=2.0
Output tag 'run3': device_run3.msh, device_run3.vtk, device_groups_run3.json
Mesher settings: order=2 (curvilinear), curvature=24/2pi, grading=on

Bodies in device.step:
  JJ              sheet  vacuum              0.001 x 0.0019 x 0
  Pin_1           solid  copper              1 x 1 x 7
  cavity          solid  vacuum              8 x 26.5 x 35
  chip            solid  sapphire            4 x 20 x 0.43
  medium_lead     sheet  vacuum              0.01 x 0.085 x 0
  pads            sheet  vacuum              1 x 1.185 x 0
  pin             solid  perfect conductor   3 x 3 x 17
  thin_lead       sheet  vacuum              0.001 x 0.025 x 0

Derived roles:
  JJ              sheet  -> junction (boundary 'JJ_inducta

## 5. Palace config
Generated entirely from the groups file + `device_config.json` — materials,
boundaries, junction port, solver block. Read the printed summary and check
it names every physical group you expect (a missing group would have raised
an error).

In [20]:
import subprocess, sys, json, os

PALACE_CONFIG = f"palace_config_{TAG}.json"
cmd = [sys.executable, "write_palace_config.py",
       "--mesh", MSH, "--groups", GROUPS,
       "--device-config", "device_config.json",
       "--output", PALACE_CONFIG]
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=RUN_DIR, check=True)

# point this run's output at its own postpro folder
path = os.path.join(RUN_DIR, PALACE_CONFIG)
pc = json.load(open(path))
pc.setdefault("Problem", {})["Output"] = f"postpro_{TAG}"
json.dump(pc, open(path, "w"), indent=2)
print("Problem.Output -> postpro_" + TAG)

$ c:\Users\qcrew5\anaconda3\envs\qiskit_metal\python.exe write_palace_config.py --mesh device_run2.msh --groups device_groups_run2.json --device-config device_config.json --output palace_config_run2.json
Problem.Output -> postpro_run2


## 6. PBS file + completeness check
Writes `run_palace.pbs` (with the crashed-run `postpro` guard built in) and
verifies the folder is internally consistent: mesh ↔ config ↔ pbs all
reference each other.

In [21]:
ph.write_pbs(RUN_DIR, PALACE_CONFIG, **HPC)
ph.final_checklist(RUN_DIR, MSH, PALACE_CONFIG)

written: //Qcrew_drive/home/vashti_enjoy/cavity_pin_loss_chip_JJ_export_20260807_150153\run_palace.pbs
Run folder is complete:
  //Qcrew_drive/home/vashti_enjoy/cavity_pin_loss_chip_JJ_export_20260807_150153
    device_run2.msh                             66.8 MB
    palace_config_run2.json                      0.0 MB
    run_palace.pbs                               0.0 MB

On the HPC (after copying the folder over):
  cd <copied folder>
  qsub run_palace.pbs

Results land in <folder>/postpro/ (eig.csv + the log).


## 7. On the HPC

Copy the whole run folder to the cluster (drag it in your file browser, or
`scp -r`). Then:

```bash
cd <the folder>
qsub run_palace.pbs
```

That's it. Watch the job with `qstat -u $USER`; the mode table appears in
the job's `.o` log file, and machine-readable results in `postpro/eig.csv`.

**Afterwards** — for field plots, add `"Save": 5` inside
`Solver.Eigenmode` of the palace config before submitting, and pull back
`postpro/paraview/` (see ParaView notes). For a mesh sweep, change `TAG`
and `SIZES` in cell 1 and re-run from cell 3 — each tag gets its own mesh,
config, and can get its own folder/PBS.